In [1]:
model_file = "resnet18.mlir.v1"
#target_file = model_file + ".recompute"
#target_file = model_file + ".storeall"
#target_file = model_file + ".hybrid"
target_file = model_file + ".heuristic"
!./convert_v0_to_v1.sh {model_file}
!ragdoll-opt {model_file} --ragdoll-identical-transpose-removal > {model_file}.tmp
!cat {model_file}.tmp

In [2]:
!ragdoll-opt {model_file}  \
--canonicalize \
--enable-cse-in-legalizer \
--symbol-dce \
--ragdoll-autodiff \
--ragdoll-autodiff-inline-function-call \
--ragdoll-initialisation \
--eliminate-empty-tensors \
--ragdoll-legalise-to-iree-compatibility \
--ragdoll-raise-linalg-to-tosa \
--canonicalize \
--cse > {target_file}

In [3]:
!cat {target_file}

#map = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1 + d4, d2 + d5, d3)>
#map1 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d4, d5)>
#map2 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1, d2, d3)>
#map3 = affine_map<(d0, d1, d2, d3, d4, d5) -> (d0, d1 * 2 + d4, d2 * 2 + d5, d3)>
module attributes {torch.debug_module_name = "ResNet"} {
  ml_program.global private mutable @global0(dense<1.000000e+00> : tensor<1x224x224x3xf32>) : tensor<1x224x224x3xf32>
  func.func @forward(%arg0: tensor<1x224x224x3xf32>) -> tensor<1x1000xf32> {
    %0 = "tosa.const"() <{value = dense<7.777000e-02> : tensor<64x7x7x3xf32>}> : () -> tensor<64x7x7x3xf32>
    %1 = "tosa.const"() <{value = dense<7.777000e-02> : tensor<1x1x1x64xf32>}> : () -> tensor<1x1x1x64xf32>
    %2 = "tosa.const"() <{value = dense<7.778000e-02> : tensor<1x1x64xf32>}> : () -> tensor<1x1x64xf32>
    %3 = "tosa.const"() <{value = dense<7.777000e-02> : tensor<64x3x3x64xf32>}> : () -> tensor<64x3x3x64xf32>
    %4 = "tosa.const"() <{value = dense

In [4]:
!ragdoll-opt --help | grep remove

      --ragdoll-remove-globals                               -   Remove `ml_program.global_store` and `ml_program.global_load` operations
      --remove-dead-values                                   -   Remove dead values
      --remove-shape-constraints                             -   Replace all cstr_ ops with a true witness
    =data-and-control-without-rt-check                       -   Similar to data-and-control, but remove the runtime check
